# 04 Snow reliability

Defines elevation bands from the master plan's own stated elevations (claim C033) and records the one historical snowfall figure available (claim C034) as the baseline a reliability indicator would be validated against. It does NOT compute a reliability indicator against baseline/mid-century climate projections: no CanDCS-M6/SWE grid cell for Revelstoke has been pulled yet (see Open issues in steps/04). Producing a plausible-looking projection number without that data would violate CLAUDE.md's rule against stating a figure with no evidence behind it, so this notebook stops at what it can actually show.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"

## Load the ledger

Confirms C033 (elevation) and C034 (historical snowfall) still point to the source this notebook reads from.

In [ ]:
import sys

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C033"]["source_id"] == "S034"
assert ledger["claims"]["C034"]["source_id"] == "S034"
print("C033, C034 confirmed against S034")

## Elevation bands (claim C033)

The master plan states five elevation points directly (bottom, lift top, sub peak, summit) rather than a continuous profile, so bands are built between those stated points instead of picking arbitrary round-number cutoffs. This also gives a free internal consistency check: lift top minus bottom should equal the plan's separately stated lift-accessed vertical.

In [ ]:
import pandas as pd

elevation_points_m = {
    "Bottom (Lower Village)": 512,
    "Lift top (The Stoke Chair)": 2225,
    "Sub Peak": 2340,
    "Mt. Mackenzie Summit": 2466,
}
bands = pd.DataFrame(
    [
        {"band": "Base to lift top", "bottom_m": 512, "top_m": 2225},
        {"band": "Lift top to sub peak", "bottom_m": 2225, "top_m": 2340},
        {"band": "Sub peak to summit", "bottom_m": 2340, "top_m": 2466},
    ]
)
bands["band_vertical_m"] = bands["top_m"] - bands["bottom_m"]

stated_lift_accessed_vertical_m = 1713
assert bands.loc[0, "band_vertical_m"] == stated_lift_accessed_vertical_m, (
    "base-to-lift-top band should match the plan's stated lift-accessed vertical"
)
bands

## Historical snowfall baseline (claim C034)

The master plan states a range, not a single figure, and no year range or measurement method for it. Both bounds are kept, and the uncertainty is written into the output rather than silently collapsed to a midpoint.

In [ ]:
historical_snowfall = {
    "rmr_annual_m_low": 9,
    "rmr_annual_m_high": 14,
    "selkirk_mountains_annual_m_low": 12,
    "selkirk_mountains_annual_m_high": 18,
}
historical_snowfall

## Reliability indicator (judgment call, not yet evaluated)

Proposes the indicator as a ratio: projected seasonal snowfall divided by the historical baseline above, per elevation band. This is a judgment call (no claim backs this specific formula); it is defined here so the method is on record, but it is not run against baseline or mid-century projections, because no CanDCS-M6 or SWE grid-cell data for Revelstoke has been retrieved (see Open issues in steps/04-snow-reliability.md, and claim C036 on the THREDDS access path found but not yet used).

In [ ]:
def snow_reliability_ratio(projected_seasonal_snowfall_m, historical_baseline_m):
    """Ratio of projected to historical seasonal snowfall. >1 means
    snowier than the historical baseline, <1 means less snowy. Not yet
    evaluated against real projections; see the markdown above."""
    return projected_seasonal_snowfall_m / historical_baseline_m


projections_available = False

## Write outputs

Elevation bands and the historical baseline are real outputs; the indicator function is recorded as a method, not a result, so nothing computed from it goes to data/processed yet.

In [ ]:
import json
import os

os.makedirs(processed_dir, exist_ok=True)
bands.to_csv(f"{processed_dir}/04_elevation_bands.csv", index=False, encoding="utf-8")
with open(f"{processed_dir}/04_historical_snowfall.json", "w", encoding="utf-8") as f:
    json.dump(historical_snowfall, f, indent=2)
print("wrote 04_elevation_bands.csv, 04_historical_snowfall.json")
print(f"projections_available: {projections_available}")

## Checks

The band boundaries must be strictly increasing and the base band's vertical must match the plan's separately stated lift-accessed vertical (already asserted above); this check also records, in the output itself, that no projection has run yet, so a downstream step cannot mistake an empty result for a zero.

In [ ]:
assert (bands["top_m"] > bands["bottom_m"]).all()
assert bands["bottom_m"].is_monotonic_increasing
assert historical_snowfall["rmr_annual_m_low"] < historical_snowfall["rmr_annual_m_high"]
assert projections_available is False, (
    "flip this only once real baseline/mid-century projection data backs it"
)
print("checks passed (elevation bands and historical baseline only; no projection run)")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))